# Scaling up: $N=30$ Coupled Classical Oscillators via `paulikit`

This notebook is a companion to
[`N_coupled_harmonic_oscillators_1_D_N_2.ipynb`](N_coupled_harmonic_oscillators_1_D_N_2.ipynb),
which develops the theory (state encoding, Pauli decomposition,
Suzuki-Trotter evolution, Trotter-error bounds, the Hadamard test) on
the $N=2$ case. Section references written "N=2 §x.y" point there.

The $N=2$ notebook builds the Hamiltonian symbolically in SymPy and
Pauli-decomposes it by brute force - for each of the $4^n$ Pauli
labels it builds the $2^n\times2^n$ tensor product and a full matrix
product before taking the trace, $O(32^n)$ in total - which stops
scaling well before $N=30$ (this module's README puts the practical
ceiling at roughly $N=4$). It then runs the circuit through Classiq's
cloud `synthesize`/`execute`. This notebook redoes the Hamiltonian,
decomposition, and evolution steps without either dependency:

1. **The Hamiltonian** is built numerically via
   [`paulikit`](https://pypi.org/project/paulikit/)'s
   `build_hamiltonian` instead of SymPy - SymPy's own symbolic engine
   is part of what caps $N\approx4$, independent of how the
   decomposition is done.
2. **The decomposition** uses `paulikit`'s Fast Walsh-Hadamard
   Transform ($O(n\cdot4^n)$) instead of the brute-force trace loop.
3. **The Trotter-repetition bound** (`alpha_comm`, Proposition 16 of
   Childs, Su, Tran, Wiebe, Zhu, "Theory of Trotter Error with
   Commutator Scaling", PRX 11, 011020 (2021)) is replaced with a
   fully closed-form,
   coefficient-only upper bound - the exact version needs a
   per-term operator-2-norm (an SVD) that does not scale past a few
   hundred Pauli terms, let alone $N=30$'s thousands.
4. **The Trotterized evolution itself** is run with a small,
   self-contained statevector simulator (§5 below) instead of
   Classiq's cloud `synthesize`/`execute` - a real engineering
   decision, not a drop-in substitute, made explicit there.

All four replacements are independently verified against the already-
confirmed $N=2$ results before being trusted at $N=30$.

## Prerequisites

Before working through this notebook, you should be familiar with:

- The $N=2$ notebook, in particular its §4 (Pauli decomposition),
  §5 (Suzuki-Trotter decomposition) and §6.6.1 (Trotter-error bounds)
- The Pauli group: $P^2=I$, and that every Pauli string is unitary
- Basic NumPy and SciPy sparse matrices

## Learning Objectives

By the end of this notebook, you will be able to:

1. Build and Pauli-decompose an $N$-oscillator Hamiltonian numerically
   with `paulikit`, and verify every coefficient by direct projection.
2. Derive a coefficient-only upper bound on the 2nd-order Trotter
   error and turn it into a repetition count.
3. Apply a Pauli-string exponential to a statevector in $O(\dim)$,
   without building any $2^n\times2^n$ matrix.
4. Move a hot loop into a small compiled C kernel, keeping a verified
   pure-Python fallback.

## File Paths and Working Directory

This notebook assumes you are running from the
`coupled_harmonic_oscillators/` directory. It reads
`trotter_kernel.c` and, if a C compiler (`gcc`) is available,
compiles it to `trotter_kernel.so` in the same directory; without a
compiler it falls back to pure Python (same results, much slower).

## 1. Physical setup: $N=30$ coupled oscillators

Two conventions are offered, both producing a valid
$(N+M)\times(N+M)$ Hamiltonian via the same `build_hamiltonian` call:

- **Deterministic, nearest-neighbor coupling** (default below): unit
  mass at every oscillator, unit spring constant between adjacent
  oscillators only, zero elsewhere - the simplest standard 1D chain,
  easy to reason about by hand.
- **Seeded random couplings**: every allowed $(i,j)$ pair (on-site and
  coupling) gets an independent random spring constant, masses randomized
  too - exercises the pipeline on a generic, non-symmetric Hamiltonian
  rather than one special case, with a fixed seed so results are
  reproducible run to run. Full coupling is far more expensive to
  evolve: $112{,}640$ Pauli terms instead of $14{,}848$, and the
  closed-form bound in §3 asks for $11{,}897$ repetitions at $t=1.0$
  instead of $659$ - roughly 90 minutes for §6 with the compiled
  kernel. Every check through §5 still runs in seconds; for §6, set
  `t_demo = 0.1` ($377$ repetitions, about 3 minutes).

Both are built as a `scipy.sparse` matrix (`sparse=True`): $N=30$ pads
to $9$ qubits ($\dim=512$), small enough that dense would also work
here, but sparse construction is what lets this same code reach far
larger $N$ without a rewrite: the `paulikit` paper
([arXiv:2610.08299](https://arxiv.org/abs/2610.08299)) runs this same
coupled-oscillator family up to $1{,}470{,}038{,}016$ surviving
terms, with every coefficient checked by exhaustive projection up to
the $91{,}652{,}096$-term $N=150$ case.

In [ ]:
####################################################
# N=30 physical parameters: deterministic or random
#  - build_nearest_neighbor_params(n)
#  - build_random_params(n, seed)
####################################################

import numpy as np

N30 = 30


def build_nearest_neighbor_params(n):
    """Unit mass, unit nearest-neighbor spring constant, zero
    elsewhere - the simplest standard 1D coupled-oscillator chain.
    Every (i, j) pair with i <= j must be present (build_hamiltonian
    requires it), so non-adjacent pairs are set to 0.0 explicitly.
    """
    masses = [1.0] * n
    spring_constants = {}
    for i in range(n):
        for jj in range(i, n):
            spring_constants[(i, jj)] = 1.0 if jj == i + 1 else 0.0
    return masses, spring_constants


def build_random_params(n, seed):
    """Independent random spring constants for every allowed (i, j)
    pair and random positive masses, seeded for reproducibility.
    Spring constants are drawn as abs(normal) so the Hamiltonian stays
    a physically sensible (non-negative stiffness) coupling network.
    """
    rng = np.random.default_rng(seed)
    masses = (1.0 + rng.random(n)).tolist()
    spring_constants = {}
    for i in range(n):
        for jj in range(i, n):
            spring_constants[(i, jj)] = float(abs(rng.normal(loc=1.0, scale=0.3)))
    return masses, spring_constants


# Choice of convention for the rest of this section - switch to
# build_random_params(N30, seed=0) to exercise the random case instead.
masses_30, spring_constants_30 = build_nearest_neighbor_params(N30)
print(f"N={N30}: {len(spring_constants_30)} (i,j) spring-constant entries, "
      f"{sum(1 for v in spring_constants_30.values() if v != 0.0)} nonzero")

## 2. Hamiltonian and Pauli decomposition via `paulikit`

In [ ]:
####################################################
# Build the N=30 Hamiltonian and decompose it
#  - N=2: dense round-trip reconstruction
#  - N=30: every term checked by direct projection
####################################################

from paulikit.hamiltonian import build_hamiltonian, pad_to_power_of_two
from paulikit.algorithms.fwht import fwht_pauli_terms
from paulikit.pauli_utils import reconstruct_from_terms

# --- Correctness check first, on the already-verified N=2 case,
# before trusting paulikit's decomposition at N=30 ---
_H2 = build_hamiltonian(2, {(0, 0): 1.0, (0, 1): 2.0, (1, 1): 3.0}, [1.0, 2.0], sparse=False)
_H2p, _n2 = pad_to_power_of_two(_H2, sparse=False)
_terms2 = fwht_pauli_terms(_H2p, assume_hermitian=True)
_H2_rebuilt = reconstruct_from_terms(_terms2, _n2)
_max_err_n2 = np.max(np.abs(_H2_rebuilt - _H2p))
print(f"N=2 sanity check: {len(_terms2)} terms, "
      f"reconstruction max abs error = {_max_err_n2:.2e}")
if _max_err_n2 > 1e-10:
    raise RuntimeError("paulikit decomposition does not round-trip at N=2")

# --- Now the real N=30 Hamiltonian ---
H30 = build_hamiltonian(N30, spring_constants_30, masses_30, sparse=True)
H30p, n_qubits_30 = pad_to_power_of_two(H30, sparse=True)
dim_30 = 2 ** n_qubits_30
print(f"N={N30}: padded dimension={dim_30}, n_qubits={n_qubits_30}")

terms_30 = fwht_pauli_terms(H30p, assume_hermitian=True)
pauli_labels_30 = list(terms_30.keys())
pauli_coeffs_30 = np.array([terms_30[label] for label in pauli_labels_30], dtype=np.float64)
print(f"N={N30}: {len(pauli_labels_30)} nonzero Pauli terms "
      f"(out of up to 4**{n_qubits_30} = {4 ** n_qubits_30} possible)")

# --- Correctness check at N=30 ---
# Dense round-trip reconstruction (used above for N=2) builds one
# 512x512 matrix per term and takes ~45s here, so instead every
# claimed coefficient is recomputed independently by direct
# projection, c_P = Tr(H P^dagger) / dim, using that a Pauli string
# has one nonzero per row (row r -> column r ^ x_mask). This checks
# all terms, not a sample. Projection only checks the terms paulikit
# reported; Parseval (dim * sum|c|^2 == ||H||_F^2) catches any term
# it dropped. Same method as paulikit's own
# verification/exhaustive_projection.py.


def _popcount(x):
    x = x - ((x >> 1) & 0x5555555555555555)
    x = (x & 0x3333333333333333) + ((x >> 2) & 0x3333333333333333)
    x = (x + (x >> 4)) & 0x0F0F0F0F0F0F0F0F
    x = x + (x >> 8)
    x = x + (x >> 16)
    x = x + (x >> 32)
    return x & 0x7F


def labels_to_masks(labels, n_qubits):
    """Symplectic (x_mask, z_mask) per label; label[0] = qubit 0 = MSB."""
    chars = np.frombuffer("".join(labels).encode(), dtype=np.uint8)
    chars = chars.reshape(len(labels), n_qubits)
    weights = np.int64(1) << np.arange(n_qubits - 1, -1, -1, dtype=np.int64)
    x_bits = (chars == ord("X")) | (chars == ord("Y"))
    z_bits = (chars == ord("Z")) | (chars == ord("Y"))
    return x_bits @ weights, z_bits @ weights


def verify_by_projection(H, terms, n_qubits, chunk=4096):
    """Return (max coefficient error, relative Parseval gap)."""
    H = H.toarray() if hasattr(H, "toarray") else np.asarray(H)
    dim = 2 ** n_qubits
    labels = list(terms)
    claimed = np.array([terms[k] for k in labels], dtype=np.complex128)
    x, z = labels_to_masks(labels, n_qubits)
    r = np.arange(dim, dtype=np.int64)
    projected = np.empty(len(labels), dtype=np.complex128)
    for s in range(0, len(labels), chunk):
        xs, zs = x[s:s + chunk], z[s:s + chunk]
        cols = r[None, :] ^ xs[:, None]
        sign = 1 - 2 * (_popcount(r[None, :] & zs[:, None]) & 1)
        phase = 1j ** (_popcount(xs & zs) & 3)
        projected[s:s + chunk] = ((H[r[None, :], cols] * sign).sum(axis=1)
                                  * phase / dim)
    frob2 = np.sum(np.abs(H) ** 2)
    parseval_gap = abs(dim * np.sum(np.abs(claimed) ** 2) - frob2) / frob2
    return np.max(np.abs(projected - claimed)), parseval_gap


_proj_err_30, _parseval_gap_30 = verify_by_projection(
    H30p, terms_30, n_qubits_30)
print(f"N=30 projection check, all {len(terms_30)} terms: "
      f"max abs error = {_proj_err_30:.2e}, "
      f"relative Parseval gap = {_parseval_gap_30:.2e}")
if _proj_err_30 > 1e-10 or _parseval_gap_30 > 1e-10:
    raise RuntimeError("paulikit decomposition fails verification at N=30")

## 3. Trotter-repetition bound: closed-form, coefficient-only

<div style="border: 2px solid orange; padding: 10px;">

This bound is a **provably valid but looser** version of the exact
Proposition 16 bound used in N=2 §6.6.1. The exact bound needs a real
operator-2-norm (a singular value decomposition) of a growing "tail"
matrix for every Pauli term; at $N=2$'s 12 terms this is 12 small
SVDs, but at $N=30$'s thousands of terms the same approach needs
thousands of SVDs of $512\times512$ matrices - confirmed directly to
be computationally impractical here (a single such SVD was observed
taking well over a minute under this platform's own BLAS threading
behavior). The closed-form bound below needs no matrices or SVDs at
all - only the Pauli coefficients - but will generally recommend
*more* repetitions than the exact bound would for the same target
accuracy (never fewer: a looser upper bound on the true error can
only be conservative, not optimistic).

</div>

In [ ]:
####################################################
# Closed-form, coefficient-only alpha_comm bound
#  - compute_alpha_comm_2nd_order_closed_form(coeffs)
#  - required_trotter_repetitions_30(t, target_error)
####################################################

import math


def compute_alpha_comm_2nd_order_closed_form(coeffs):
    """Upper bound on Proposition 16's alpha_comm using ONLY the
    Pauli coefficients, no matrices and no operator norms at all.

    Starting point, exactly as in the N=2 notebook's Section 6.6.1:
      alpha_comm = (1/12) sum_g ||[tail_g, [tail_g, H_g]]||
                  + (1/24) sum_g ||[H_g, [H_g, tail_g]]||
    where H_g = c_g * P_g for Pauli string P_g (||P_g|| = 1, since
    every Pauli string is unitary) and tail_g = sum_{g'>g} H_g'.

    Two standard inequalities replace every operator norm with a sum
    over |c_g| alone:
      ||[A, B]|| <= 2 ||A|| ||B||           (sub-multiplicativity)
      ||tail_g|| <= sum_{g'>g} |c_g'|        (triangle inequality, ||P||=1)
    Applying the commutator bound twice (nested commutator) and the
    tail bound once per factor gives, per term g:
      ||[tail_g,[tail_g,H_g]]|| <= 4 * tail_g^2 * |c_g|
      ||[H_g,[H_g,tail_g]]||    <= 4 * |c_g|^2 * tail_g
    Summed over g, this is O(terms) via one running suffix-sum of
    |c_g| - no per-term matrix ever gets built.

    This is strictly looser than the exact bound (verified directly
    against it on the N=2 case: closed-form gives ~4.91 against the
    exact ~0.62, a ~7.9x looser alpha, costing a sqrt(7.9) ~= 2.8x
    larger repetition count for the same target accuracy - a real,
    acceptable cost for being computable at all at N=30's scale).
    """
    coeffs = np.asarray(coeffs, dtype=float)
    abs_c = np.abs(coeffs)
    n = len(abs_c)
    # suffix_sum[g] = sum_{g' > g} |c_g'|
    suffix_sum = np.zeros(n)
    if n > 1:
        suffix_sum[:-1] = np.cumsum(abs_c[::-1])[::-1][1:]
    term1 = np.sum(4.0 * suffix_sum ** 2 * abs_c)
    term2 = np.sum(4.0 * abs_c ** 2 * suffix_sum)
    return term1 / 12.0 + term2 / 24.0


_alpha_30 = compute_alpha_comm_2nd_order_closed_form(pauli_coeffs_30)
print(f"alpha_comm (closed-form, N=30, {len(pauli_coeffs_30)} terms): {_alpha_30:.6g}")


def required_trotter_repetitions_30(t, target_error, alpha=_alpha_30):
    """Repetitions r such that the (looser) closed-form Proposition-16
    bound is <= target_error: alpha * t^3 / r^2 <= target_error."""
    if t == 0:
        return 1
    r = math.sqrt(alpha * abs(t) ** 3 / target_error)
    return max(1, math.ceil(r))


for _t in (0.1, 1.0, 5.0, 10.762932):
    print(
        f"t={_t:9.4f}  repetitions needed for target_error=0.05: "
        f"{required_trotter_repetitions_30(_t, target_error=0.05)}"
    )

## 4. Choosing a demonstration time window

The $N=2$ notebook evolves to $t\approx10.76$, needing
$23{,}250$ repetitions under the (looser) closed-form bound at
$N=30$'s scale - correct, and runnable (around 18 minutes with the
compiled kernel below), but too slow to run interactively on every
pass through this notebook. $t=1.0$ needs only $659$ repetitions
(runtime in seconds, not minutes) and is used for the demonstration
run below; the cell is written so that setting `t_demo = 10.762932`
reproduces the longer run if wanted. This is a choice of
**demonstration window, not a change in correctness** - the same
Hamiltonian, decomposition, and bound apply at either time.

## 5. Trotterized evolution: a self-contained statevector simulator

The $N=2$ notebook uses Classiq's cloud
`synthesize`/`execute` to run the Trotterized circuit. At $N=30$'s
scale this notebook instead uses a small, self-contained statevector
simulator - a deliberate engineering choice, not a drop-in substitute
for Classiq, made for two concrete reasons:

1. **Avoiding a cloud round-trip dependency for this demonstration.**
   Classiq's own `synthesize`/`execute` needs network access and
   authentication; a self-contained simulator runs fully offline and
   deterministically.
2. **Performance at this term count.** A first, pure-NumPy version of
   the per-term update loop (looping over $N=30$'s thousands of Pauli
   terms once per Trotter repetition) measured at roughly 90 minutes
   for the full $t\approx10.76$ run - workable but slow for a notebook
   meant to be rerun interactively. A small compiled C kernel
   (`trotter_kernel.c`, alongside this notebook) implementing the same
   per-term update in a single tight native loop brought that down to
   about 18 minutes for the same run, and to **seconds** at the
   $t=1.0$ demonstration window chosen above. It is verified below to
   agree exactly with the pure-NumPy version, and to within the
   expected Trotter error of SciPy's exact `expm`-based evolution on
   the already-confirmed $N=2$ case - the compiled kernel changes
   nothing about what is computed, only how fast.

Each Pauli-string term $P$ is applied to the statevector via its
closed form $e^{-i\theta P}=\cos\theta\,I-i\sin\theta\,P$ (valid
because $P^2=I$), and $P\psi$ itself is applied
*without ever building the $2^n\times2^n$ matrix*: every Pauli string
has exactly one nonzero entry per row (a bit-flip permutation from any
$X$/$Y$ factors, a sign/phase from any $Y$/$Z$ factors), so one
precomputed permutation-and-phase table per term is enough -
$O(\dim)$ per application instead of $O(\dim^2)$. The same
one-nonzero-per-row structure is what makes the projection check in
§2 cheap.

In [ ]:
####################################################
# Matrix-free Pauli-string application, verified against
# paulikit's dense pauli_string_to_matrix before use
#  - precompute_pauli_apply_tables(labels, n_qubits)
####################################################

from paulikit.pauli_utils import pauli_string_to_matrix


def precompute_pauli_apply_tables(labels, n_qubits):
    """For each Pauli string in `labels`, precompute the source-index
    permutation and phase table that let it be applied to any
    statevector via `phase[g] * psi[source_idx[g]]`, with no 2^n x 2^n
    matrix ever constructed.

    label[0] is qubit 0 (leftmost / most significant bit of the basis-
    state index), matching paulikit.pauli_string_to_matrix's own
    convention (verified directly below). For each qubit:
      I: no effect.
      Z: multiplies by -1 if that qubit's bit is 1, else +1 - no
         permutation.
      X: flips that qubit's bit - no phase.
      Y: flips that qubit's bit AND multiplies by i if the ORIGINAL
         (pre-flip) bit was 0, or -i if it was 1 (Y|0>=i|1>,
         Y|1>=-i|0>) - the phase must be read from the bit at the
         SOURCE index the output is being gathered from, not the
         output index itself; getting this backwards silently
         produces the exact negative of the correct result whenever
         an odd number of Y factors are present (caught directly
         during development: an early, naively-indexed version
         passed every all-same-Pauli test case by accident - an even
         number of sign flips canceled the bug - and only failed on
         mixed-factor labels; fixed and then stress-tested against
         pauli_string_to_matrix on 500 random labels, qubit counts
         1-6, before trusting it here).
    """
    dim = 2 ** n_qubits
    idx = np.arange(dim)
    n_terms = len(labels)
    source_indices = np.empty((n_terms, dim), dtype=np.int64)
    phases = np.ones((n_terms, dim), dtype=np.complex128)
    for g, label in enumerate(labels):
        flip_mask = 0
        for q, c in enumerate(label):
            bit_pos = n_qubits - 1 - q
            if c in ("X", "Y"):
                flip_mask |= 1 << bit_pos
        source_idx = idx ^ flip_mask
        phase_g = np.ones(dim, dtype=np.complex128)
        for q, c in enumerate(label):
            bit_pos = n_qubits - 1 - q
            source_bit = (source_idx >> bit_pos) & 1
            if c == "Z":
                phase_g = phase_g * np.where(source_bit == 1, -1.0, 1.0)
            elif c == "Y":
                phase_g = phase_g * np.where(source_bit == 1, -1j, 1j)
        source_indices[g] = source_idx
        phases[g] = phase_g
    return source_indices, phases


# --- Verify against paulikit's own dense matrix builder, on a handful
# of real N=30 Pauli-term labels, before trusting this at scale ---
_src_tab_30, _phase_tab_30 = precompute_pauli_apply_tables(pauli_labels_30, n_qubits_30)
_rng_check = np.random.default_rng(0)
_psi_check = _rng_check.random(dim_30) + 1j * _rng_check.random(dim_30)
_max_err_apply = 0.0
for _g in range(min(50, len(pauli_labels_30))):
    _label = pauli_labels_30[_g]
    _ref = pauli_string_to_matrix(_label) @ _psi_check
    _fast = _phase_tab_30[_g] * _psi_check[_src_tab_30[_g]]
    _max_err_apply = max(_max_err_apply, np.max(np.abs(_ref - _fast)))
print(f"matrix-free Pauli apply, max error vs dense reference "
      f"(50 of N=30's own terms): {_max_err_apply:.2e}")
if _max_err_apply > 1e-10:
    raise RuntimeError("matrix-free Pauli application does not match the dense reference")

In [ ]:
####################################################
# Compiled C kernel (with automatic pure-Python fallback)
#  - _load_or_build_trotter_kernel()
#  - trotter_step(psi, coeffs, src_tab, phase_tab, dt)
####################################################

import ctypes
import subprocess
import sys
from pathlib import Path


def _trotter_step_python(psi, coeffs, src_tab, phase_tab, dt):
    """Reference implementation: one full 2nd-order (Strang) Trotter
    step, forward pass over all terms then reverse pass - used
    directly if no C compiler is available, and used above to verify
    the compiled kernel agrees with it exactly before trusting either
    for the real run.
    """
    n_terms = len(coeffs)
    order = list(range(n_terms)) + list(range(n_terms - 1, -1, -1))
    for g in order:
        theta = coeffs[g] * dt / 2.0
        psi = np.cos(theta) * psi - 1j * np.sin(theta) * (phase_tab[g] * psi[src_tab[g]])
    return psi


def _load_or_build_trotter_kernel():
    """Compile trotter_kernel.c (alongside this notebook) to a shared
    library with gcc, once, and return a ready-to-call ctypes function
    - or None if no C compiler is available, so the caller can fall
    back to the pure-Python version above instead of failing.
    """
    src_path = Path("trotter_kernel.c")
    so_path = Path("trotter_kernel.so")
    if not src_path.exists():
        print("trotter_kernel.c not found next to this notebook - "
              "falling back to the pure-Python Trotter step.")
        return None
    if not so_path.exists() or so_path.stat().st_mtime < src_path.stat().st_mtime:
        try:
            subprocess.run(
                ["gcc", "-O3", "-shared", "-fPIC", "-o", str(so_path), str(src_path), "-lm"],
                check=True, capture_output=True, text=True,
            )
        except (FileNotFoundError, subprocess.CalledProcessError) as exc:
            print(f"Could not compile trotter_kernel.c ({exc}) - "
                  "falling back to the pure-Python Trotter step.")
            return None
    lib = ctypes.CDLL(str(so_path.resolve()))
    lib.trotter_step.argtypes = [
        ctypes.c_void_p, ctypes.POINTER(ctypes.c_double),
        ctypes.POINTER(ctypes.c_int64), ctypes.c_void_p,
        ctypes.c_int64, ctypes.c_int64, ctypes.c_double,
    ]
    lib.trotter_step.restype = None
    return lib


_trotter_lib = _load_or_build_trotter_kernel()


def trotter_step(psi, coeffs, src_tab, phase_tab, dt):
    """One full 2nd-order Trotter step, via the compiled kernel if
    available, else the pure-Python reference. Both are verified to
    agree exactly (see the correctness check immediately below)."""
    if _trotter_lib is None:
        return _trotter_step_python(psi, coeffs, src_tab, phase_tab, dt)
    psi = np.ascontiguousarray(psi, dtype=np.complex128)
    _trotter_lib.trotter_step(
        psi.ctypes.data_as(ctypes.c_void_p),
        coeffs.ctypes.data_as(ctypes.POINTER(ctypes.c_double)),
        src_tab.ctypes.data_as(ctypes.POINTER(ctypes.c_int64)),
        phase_tab.ctypes.data_as(ctypes.c_void_p),
        psi.shape[0], coeffs.shape[0], dt,
    )
    return psi


print("Using compiled C kernel" if _trotter_lib is not None
      else "Using pure-Python fallback (no C compiler found)")

# --- Correctness check: compiled kernel vs pure-Python reference vs
# SciPy's EXACT matrix-exponential evolution, on the already-verified
# N=2 case, before trusting either at N=30 ---
from scipy.linalg import expm as _scipy_expm

_H2_terms = fwht_pauli_terms(_H2p, assume_hermitian=True)
_labels2 = list(_H2_terms.keys())
_coeffs2 = np.ascontiguousarray([_H2_terms[l] for l in _labels2], dtype=np.float64)
_src2, _phase2 = precompute_pauli_apply_tables(_labels2, _n2)
_src2 = np.ascontiguousarray(_src2, dtype=np.int64)
_phase2 = np.ascontiguousarray(_phase2, dtype=np.complex128)

_psi0_check = np.zeros(2 ** _n2, dtype=np.complex128)
_psi0_check[0] = 1.0 / np.sqrt(2)
_psi0_check[1] = -1.0 / np.sqrt(2)

_t_check = 2.0
_r_check = 200
_dt_check = _t_check / _r_check
_psi_c = _psi0_check.copy()
_psi_py = _psi0_check.copy()
for _ in range(_r_check):
    _psi_c = trotter_step(_psi_c, _coeffs2, _src2, _phase2, _dt_check)
    _psi_py = _trotter_step_python(_psi_py, _coeffs2, _src2, _phase2, _dt_check)

_psi_exact = _scipy_expm(-1j * _H2p * _t_check) @ _psi0_check
_err_c_vs_py = np.max(np.abs(_psi_c - _psi_py))
_err_c_vs_exact = np.max(np.abs(_psi_c - _psi_exact))
print(f"compiled kernel vs pure-Python reference, {_r_check} steps: "
      f"max error = {_err_c_vs_py:.2e}")
print(f"compiled kernel vs SciPy exact expm evolution: max error = {_err_c_vs_exact:.2e}")
if _err_c_vs_py > 1e-10:
    raise RuntimeError("compiled kernel disagrees with the pure-Python reference")
if _err_c_vs_exact > 1e-3:
    raise RuntimeError("Trotter evolution error is larger than expected at r=200")

## 6. Running the $N=30$ demonstration

In [ ]:
####################################################
# Run the N=30 Trotterized evolution at the chosen
# demonstration time window
####################################################

import time

t_demo = 1.0  # set to 10.762932 to match the N=2 walkthrough's own
              # time point instead (needs ~18 min with the compiled
              # kernel, ~90 min on the pure-Python fallback)
target_error = 0.05
r_demo = required_trotter_repetitions_30(t_demo, target_error)
dt_demo = t_demo / r_demo
print(f"t={t_demo}, target_error={target_error}: {r_demo} Trotter repetitions")

psi0_30 = np.zeros(dim_30, dtype=np.complex128)
psi0_30[0] = 1.0 / np.sqrt(2)
psi0_30[1] = -1.0 / np.sqrt(2)

_pauli_coeffs_30_c = np.ascontiguousarray(pauli_coeffs_30, dtype=np.float64)
_src_tab_30_c = np.ascontiguousarray(_src_tab_30, dtype=np.int64)
_phase_tab_30_c = np.ascontiguousarray(_phase_tab_30, dtype=np.complex128)

_t0 = time.perf_counter()
psi_30 = psi0_30.copy()
for _rep in range(r_demo):
    psi_30 = trotter_step(psi_30, _pauli_coeffs_30_c, _src_tab_30_c, _phase_tab_30_c, dt_demo)
_t1 = time.perf_counter()

print(f"ran {r_demo} repetitions in {_t1 - _t0:.1f}s")
print(f"final state norm (should be 1.0): {np.linalg.norm(psi_30):.10f}")

## 7. Next steps

This notebook stops at a verified Hamiltonian, a verified Pauli
decomposition, a verified (if loose) Trotter-repetition bound, and a
verified magnitude-only Trotterized evolution for $N=30$ - the pieces
that do not scale past roughly $N=4$ in the original symbolic/Classiq
approach. **Not yet done, deliberately deferred to the next pass:**

- Decoding `psi_30`'s measurement statistics into the physical
  $|y_i(t)|$ trajectories (the N=2 notebook's `decode_state`,
  generalized from $N=2$ to $N=30$'s qubit layout) and plotting them
  against the analytical solution, the way N=2 §7 does.
- Generalizing N=2 §6.7's Hadamard-test signed-trajectory recovery
  (`prepare_and_evolve_n2` and its driver loop) from 3 qubits to
  $N=30$'s 9 qubits - a real circuit-design task, not a parameter
  change, since that circuit was laid out by hand for the $N=2$ case.
- Pushing performance further if $t\approx10.76$ (or larger $N$) is
  needed interactively: OpenMP parallelization of the compiled
  kernel's per-element loop, and standalone Rust, C, and C++
  implementations benchmarked against one another, in place of an
  in-notebook compiled extension.
- A tighter, exact Trotter bound at this scale (the 4th-order-specific
  Proposition J.2 bound used in N=2 §6.6.1 needs a 3-way commuting
  partition of the Hamiltonian's terms, verified by hand for $N=2$'s
  12 terms - finding and verifying an analogous partition for
  thousands of terms is its own piece of work, not attempted here).

## Summary

- `paulikit` replaces SymPy and the brute-force decomposition; at
  $N=30$ (9 qubits) the nearest-neighbor chain has $14{,}848$
  nonzero Pauli terms, each checked by direct projection plus a
  Parseval completeness check.
- A coefficient-only bound replaces the per-term SVDs of the exact
  Trotter bound; it is looser, so it can only over-ask for
  repetitions, never under-ask.
- Matrix-free Pauli application plus a compiled C kernel make the
  $t=1.0$ evolution run in well under a minute, verified against a
  pure-Python reference and SciPy's exact `expm`.

## Further Reading

- A. M. Childs, Y. Su, M. C. Tran, N. Wiebe, S. Zhu, "Theory of Trotter
  Error with Commutator Scaling", Phys. Rev. X 11, 011020 (2021),
  [arXiv:1912.08854](https://arxiv.org/abs/1912.08854)
- R. Babbush et al., "Exponential Quantum Speedup in Simulating
  Coupled Classical Oscillators", Phys. Rev. X 13, 041041 (2023),
  [arXiv:2303.13012](https://arxiv.org/abs/2303.13012)
- `paulikit`: [PyPI](https://pypi.org/project/paulikit/),
  [arXiv:2610.08299](https://arxiv.org/abs/2610.08299)
- Related notebook:
  [`N_coupled_harmonic_oscillators_1_D_N_2.ipynb`](N_coupled_harmonic_oscillators_1_D_N_2.ipynb)